# Lab: Linear Regression on Weather Data

**Dataset:** [Weather in World War Two](https://www.kaggle.com/datasets/smid80/weatherww2/data) (Kaggle) -- daily weather station readings, including `MinTemp_C` (daily minimum temperature) and `MaxTemp_C` (daily maximum temperature), in Celsius.

**The question this lab answers:**
> Is there a relationship between the daily minimum and maximum temperature? Can you predict the maximum temperature given the minimum temperature?

This is a natural fit for **linear regression**: one numeric input (`MinTemp_C`), one numeric target (`MaxTemp_C`), and a real physical reason to expect a relationship -- a colder night is usually followed by a cooler day, and vice versa.

**How to use this notebook:**
1. Download `Summary of Weather.csv` from the Kaggle link above (you'll need a free Kaggle account; use the "Download" button on the page, or `kagglehub.dataset_download("smid80/weatherww2")` if you're working in Colab).
2. Put the CSV somewhere accessible and set `FILE_PATH` in the config cell below.
3. Work through the sections in order. Code cells marked `# TODO` are for you to fill in -- everything else is provided to keep the lab focused on the regression itself.
4. At the end, you'll export your model's coefficient and intercept to a small file that the standalone `crosscheck_regression.py` script (in the same folder) can check independently.

**What this lab assumes:** you're comfortable with the basic pandas inspection habits from notebook 02 and the train/test-split pattern from notebook 03. This lab focuses specifically on *simple* linear regression (one feature, one target) rather than the general modelling workflow.

## 0. Setup and Config

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
plt.rcParams["figure.figsize"] = (7, 4.5)

print("Environment ready.")

In [ ]:
# =========================== EDIT THIS CELL ===========================

# Point this at the real "Summary of Weather.csv" from the Kaggle dataset once
# you've downloaded it. A small sample with the same column names is provided
# so you can test the notebook's structure before the real download finishes.
FILE_PATH = "data/weatherww2_sample.csv"

# The real dataset's columns (this is what you should see once loaded):
FEATURE_COLUMN = "MinTemp_C"   # our X: the input we predict FROM
TARGET_COLUMN = "MaxTemp_C"    # our y: the value we're trying to predict

# ========================================================================

print(f"File: {FILE_PATH!r}")
print(f"Feature (X): {FEATURE_COLUMN!r}  ->  Target (y): {TARGET_COLUMN!r}")

## 1. Load and Inspect

Same habit as always (Part II, slide 24) -- load it, then look at it. Load the CSV and check its shape, a preview, and the two columns we care about.

In [ ]:
df = pd.read_csv(FILE_PATH)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

In [ ]:
# TODO: check that FEATURE_COLUMN and TARGET_COLUMN actually exist in df, and that
# both are numeric (not text). Hint: df.info(), or df[[FEATURE_COLUMN, TARGET_COLUMN]].dtypes


## 2. Check for Missing Values and Prepare the Two Columns

We only need `FEATURE_COLUMN` and `TARGET_COLUMN` for this lab -- the rest of the dataset's columns are irrelevant to this specific question, so it's fine to work with just these two.

In [ ]:
# TODO: keep only the two columns we need, then check for missing values.
# Hint:
# weather = df[[FEATURE_COLUMN, TARGET_COLUMN]].copy()
# print(weather.isna().sum())


In [ ]:
# TODO: if there are any missing values in either column, drop those rows --
# you can't train or evaluate on a row where the true answer is unknown
# (same rule as notebook 03's target-missing check).
# Hint: weather = weather.dropna().reset_index(drop=True)


## 3. Is There a Relationship? (Visualize + Correlate)

Before fitting any model, answer the first half of the lab's question visually and numerically -- a scatter plot shows the *shape* of the relationship (Part II, slide 25: "do these two variables move together?"), and the correlation coefficient puts a number on how strong it is.

- Correlation near **+1**: strong positive relationship (as one goes up, so does the other).
- Correlation near **0**: no linear relationship.
- Correlation near **-1**: strong negative relationship.

In [ ]:
# TODO: make a scatter plot with FEATURE_COLUMN on the x-axis and TARGET_COLUMN on the y-axis.
# Hint:
# fig, ax = plt.subplots()
# ax.scatter(weather[FEATURE_COLUMN], weather[TARGET_COLUMN], alpha=0.4, s=15)
# ax.set_xlabel(FEATURE_COLUMN)
# ax.set_ylabel(TARGET_COLUMN)
# ax.set_title(f"{TARGET_COLUMN} vs. {FEATURE_COLUMN}")
# plt.show()


In [ ]:
# TODO: compute the correlation coefficient between the two columns.
# Hint: correlation = weather[FEATURE_COLUMN].corr(weather[TARGET_COLUMN])
#       print(f"Correlation: {correlation:.3f}")


**Stop and answer (in your own words, in a new markdown cell below):** Based on the scatter plot and correlation coefficient, is there a relationship between minimum and maximum daily temperature? How strong is it, and does the direction (positive/negative) make physical sense?

*Your answer here.*

## 4. Train/Test Split

Same rule as always (Part II, slide 20): split before fitting anything. Since we only have one feature here, `X` will be a single-column matrix -- scikit-learn still expects it in 2D shape (`[[value], [value], ...]`, not a flat list), which is why we use `weather[[FEATURE_COLUMN]]` (double brackets) rather than `weather[FEATURE_COLUMN]` (single brackets).

In [ ]:
# TODO: build X (2D, double brackets) and y (1D, single brackets), then split
# into train/test with test_size=0.2 and random_state=RANDOM_STATE.
# Hint:
# X = weather[[FEATURE_COLUMN]]
# y = weather[TARGET_COLUMN]
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=RANDOM_STATE
# )
# print(f"Train: {len(X_train)}   Test: {len(X_test)}")


## 5. Fit a Linear Regression Model

`LinearRegression` finds the straight line `y = coef * X + intercept` that best fits the training data (minimizing squared error). Fit it on the training set only.

In [ ]:
# TODO: create a LinearRegression model and fit it on the training data.
# Hint:
# model = LinearRegression()
# model.fit(X_train, y_train)


In [ ]:
# TODO: print the fitted coefficient and intercept, and write out the equation
# in the form: MaxTemp_C = coef * MinTemp_C + intercept
# Hint:
# coef = model.coef_[0]
# intercept = model.intercept_
# print(f"{TARGET_COLUMN} = {coef:.4f} * {FEATURE_COLUMN} + {intercept:.4f}")


## 6. Evaluate on the Test Set

Now the second half of the lab's question: *can* we predict max temperature from min temperature, and how good is that prediction? Use the held-out test set (Part I, slide 14 -- regression metrics: MAE, RMSE, R²).

In [ ]:
# TODO: predict on X_test, then compute and print MAE, RMSE, and R^2.
# Hint:
# y_pred = model.predict(X_test)
# mae = mean_absolute_error(y_test, y_pred)
# rmse = mean_squared_error(y_test, y_pred) ** 0.5
# r2 = r2_score(y_test, y_pred)
# print(f"MAE:  {mae:.3f} degrees C")
# print(f"RMSE: {rmse:.3f} degrees C")
# print(f"R^2:  {r2:.3f}")


**What these mean:**
- **MAE** (mean absolute error): on average, how many degrees off is a typical prediction.
- **RMSE** (root mean squared error): similar to MAE but penalizes large errors more heavily.
- **R²**: the share of the variation in max temperature that min temperature explains, from 0 (no explanatory power) to 1 (perfect fit).

Now visualize the fitted line against the actual test data -- this is the most direct answer to "can you predict the maximum temperature given the minimum temperature?"

In [ ]:
# TODO: plot the test set as points, and the model's fitted line on top.
# Hint:
# fig, ax = plt.subplots()
# ax.scatter(X_test, y_test, alpha=0.4, s=15, label="actual")
# ax.plot(X_test, y_pred, color="red", linewidth=2, label="predicted (fitted line)")
# ax.set_xlabel(FEATURE_COLUMN)
# ax.set_ylabel(TARGET_COLUMN)
# ax.set_title("Linear regression fit on the test set")
# ax.legend()
# plt.show()


**If your line looks like a jagged zig-zag instead of a straight line:** `plt.plot` draws points in the order they appear in the data, and `X_test` isn't sorted after `train_test_split`. Sort by `X_test` first before plotting the line: `order = X_test[FEATURE_COLUMN].argsort()`, then index both `X_test` and `y_pred` with `order` before plotting. (A pure scatter plot for `y_pred` instead of `.plot()` also avoids this, at the cost of not looking like a continuous line.)

## 7. Try It: Predict a New Value

Use the fitted model to answer a concrete version of the lab's question: if tomorrow's minimum temperature is a given value, what does the model predict for the maximum?

In [ ]:
# TODO: pick a few sample MinTemp_C values (e.g. 0, 10, 20) and predict MaxTemp_C for each.
# Remember model.predict() needs a 2D input, even for one value: [[0], [10], [20]]
# Hint:
# sample_min_temps = pd.DataFrame({FEATURE_COLUMN: [0, 10, 20]})
# predictions = model.predict(sample_min_temps)
# for min_t, max_t in zip(sample_min_temps[FEATURE_COLUMN], predictions):
#     print(f"MinTemp_C = {min_t:>5.1f}  ->  predicted MaxTemp_C = {max_t:.1f}")


## 8. Export for Cross-Checking

This lab folder includes a standalone script, `crosscheck_regression.py`, that recomputes this same regression **independently** -- using plain NumPy math instead of scikit-learn -- and compares its answer to yours. This is a good habit generally: when a result matters, verifying it a second way (different tool, different method) catches mistakes that re-reading your own code won't.

Run the cell below once you've filled in Sections 4-6 above. It saves your model's coefficient, intercept, and the file path/columns you used, so the script can load the same data and compare.

In [ ]:
import json

results = {
    "file_path": FILE_PATH,
    "feature_column": FEATURE_COLUMN,
    "target_column": TARGET_COLUMN,
    "sklearn_coef": float(model.coef_[0]),
    "sklearn_intercept": float(model.intercept_),
    "sklearn_r2_test": float(r2),
}

with open("lab_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved lab_results.json:")
print(json.dumps(results, indent=2))
print("\nNow run: python crosscheck_regression.py")

## Summary

You answered both parts of the lab's question:

1. **Is there a relationship?** -- checked visually (scatter plot) and numerically (correlation coefficient) in Section 3, before fitting any model.
2. **Can you predict it?** -- fit a `LinearRegression` model in Section 5, evaluated it honestly on held-out test data in Section 6 (MAE, RMSE, R²), and used it to predict new values in Section 7.

**Now run the cross-check:** open a terminal in this `notebooks/` folder and run:
```bash
python crosscheck_regression.py
```
It reloads the same CSV, computes the regression line from scratch with NumPy (no scikit-learn), and tells you whether its coefficient/intercept match what this notebook found. If they match closely, that's independent confirmation your notebook's regression is correct -- not just internally consistent with itself.